In [ ]:
%load_ext autoreload
%autoreload 2

# MH-Weed16: YOLO11 Object Detection

Multiclass weed detection on the MH-Weed16 dataset using Ultralytics YOLO11 Nano (`yolo11n.pt`).
Accelerated for Intel Arc Graphics (`torch.xpu`), CUDA, or CPU.

## 1. Setup & Environment

In [ ]:
import os
import random
import pathlib
import subprocess
from collections import Counter

import cv2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import torch
import yaml
from ultralytics import YOLO

# Set seeds
random.seed(42)
np.random.seed(42)
torch.manual_seed(42)

# Compute device
if hasattr(torch, 'xpu') and torch.xpu.is_available():
    device = 'xpu'
    device_name = torch.xpu.get_device_name(0)
elif torch.cuda.is_available():
    device = '0'
    device_name = torch.cuda.get_device_name(0)
else:
    device = 'cpu'
    device_name = 'CPU'

print(f"PyTorch version: {torch.__version__}")
print(f"Selected device: {device} ({device_name})")

ROOT_DIR = pathlib.Path(".").resolve()

## 2. Dataset Preparation
Set up directory links pointing directly to `MH-Weed16` (no duplicate files copied), create train/val/test splits (80/10/10), and write `mhweed16.yaml`.

In [ ]:
DATA_DIR = ROOT_DIR / "data"
DATA_DIR.mkdir(exist_ok=True)

RAW_IMAGES = ROOT_DIR / "MH-Weed16" / "Crop with Weeds" / "intel Real Sense Depth_Clicks" / "intel Real Sense Depth_Clicks"
RAW_LABELS = ROOT_DIR / "MH-Weed16" / "Crop with Weeds" / "intel Real Sense Depth_Annotations" / "intel Real Sense Depth_Annotations" / "YOLO_darknet"

IMAGES_DIR = DATA_DIR / "images"
LABELS_DIR = DATA_DIR / "labels"

# Setup cross-platform junctions/symlinks
if not IMAGES_DIR.exists():
    if os.name == 'nt':
        cmd = f'cmd /c mklink /J "{IMAGES_DIR}" "{RAW_IMAGES}"'
        subprocess.run(cmd, shell=True, check=True)
    else:
        IMAGES_DIR.symlink_to(RAW_IMAGES)

if not LABELS_DIR.exists():
    if os.name == 'nt':
        cmd = f'cmd /c mklink /J "{LABELS_DIR}" "{RAW_LABELS}"'
        subprocess.run(cmd, shell=True, check=True)
    else:
        LABELS_DIR.symlink_to(RAW_LABELS)

image_files = sorted([p for p in IMAGES_DIR.glob('*.*') if p.suffix.lower() in ('.jpg', '.jpeg', '.png')])
ann_files = sorted(list(LABELS_DIR.glob('*.txt')))

print(f"Total images: {len(image_files):,}")
print(f"Total labels: {len(ann_files):,}")

# Split 80 / 10 / 10
all_paths = [os.path.normpath(str(p)) for p in image_files]
random.shuffle(all_paths)

n = len(all_paths)
n_train = int(0.80 * n)
n_val = int(0.10 * n)

train_paths = all_paths[:n_train]
val_paths = all_paths[n_train:n_train + n_val]
test_paths = all_paths[n_train + n_val:]

def write_split(path, paths):
    with open(path, 'w', encoding='utf-8') as f:
        f.write('\n'.join(paths) + '\n')

write_split(ROOT_DIR / "train.txt", train_paths)
write_split(ROOT_DIR / "val.txt", val_paths)
write_split(ROOT_DIR / "test.txt", test_paths)

print(f"Splits created: Train={len(train_paths)}, Val={len(val_paths)}, Test={len(test_paths)}")

# Species names from MH-Weed16
CLASS_NAMES = {
    0: "kena",
    1: "lavhala",
    2: "lambs_quarters",
    3: "little_mallow",
    4: "moti_dudhi",
    5: "obscure_morning_glory",
    6: "asian_pigeonwings",
    7: "bilayat",
    8: "choti_dudhi",
    9: "digitaria",
    10: "gajar_gavat",
    11: "graceful_sandmat",
    12: "sicklepod",
    13: "harali",
    14: "dwarf_cassia"
}

yaml_content = {
    'path': str(ROOT_DIR).replace('\\', '/'),
    'train': 'train.txt',
    'val': 'val.txt',
    'test': 'test.txt',
    'names': CLASS_NAMES
}

with open(ROOT_DIR / "mhweed16.yaml", 'w', encoding='utf-8') as f:
    yaml.dump(yaml_content, f, default_flow_style=False, sort_keys=False)

## 3. Exploratory Data Analysis
Inspect class balance and box geometry across annotations.

In [ ]:
box_records = []
class_counts = Counter()

for ann_path in ann_files:
    with open(ann_path, 'r', encoding='utf-8') as f:
        for line in f:
            parts = line.strip().split()
            if not parts:
                continue
            cls_id = int(parts[0])
            xc, yc, w, h = map(float, parts[1:5])
            class_counts[cls_id] += 1
            box_records.append({
                'class_id': cls_id,
                'name': CLASS_NAMES.get(cls_id, str(cls_id)),
                'width': w,
                'height': h,
                'area': w * h
            })

df_boxes = pd.DataFrame(box_records)
print(f"Total bounding boxes: {len(df_boxes):,}")
print(f"Classes present: {len(class_counts)}")

# Plot class distribution
plt.figure(figsize=(12, 4))
order = [CLASS_NAMES[i] for i in range(len(CLASS_NAMES)) if i in class_counts]
sns.countplot(data=df_boxes, x='name', order=order, color='#2c7bb6')
plt.title("Instance Count per Weed Species")
plt.xlabel("Species")
plt.ylabel("Count")
plt.xticks(rotation=40, ha='right')
plt.grid(axis='y', linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()

In [ ]:
# Bounding box geometry
fig, axes = plt.subplots(1, 3, figsize=(14, 3.8))

sns.histplot(df_boxes['width'], bins=35, kde=True, ax=axes[0], color='#2c7bb6')
axes[0].set_title("Normalized Box Width")
axes[0].set_xlabel("Width")

sns.histplot(df_boxes['height'], bins=35, kde=True, ax=axes[1], color='#1a9641')
axes[1].set_title("Normalized Box Height")
axes[1].set_xlabel("Height")

sns.histplot(df_boxes['area'], bins=35, kde=True, ax=axes[2], color='#d7191c')
axes[2].set_title("Normalized Box Area")
axes[2].set_xlabel("Area")

plt.tight_layout()
plt.show()

In [ ]:
# Visualize sample ground truth image
def show_sample(img_path, ann_path):
    img = cv2.imread(str(img_path))
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    h, w, _ = img.shape
    
    with open(ann_path, 'r', encoding='utf-8') as f:
        for line in f:
            parts = line.strip().split()
            if not parts:
                continue
            cls_id = int(parts[0])
            xc, yc, bw, bh = map(float, parts[1:5])
            
            x1 = int((xc - bw / 2) * w)
            y1 = int((yc - bh / 2) * h)
            x2 = int((xc + bw / 2) * w)
            y2 = int((yc + bh / 2) * h)
            
            cv2.rectangle(img, (x1, y1), (x2, y2), (0, 255, 0), 2)
            name = CLASS_NAMES.get(cls_id, str(cls_id))
            cv2.putText(img, name, (x1, max(20, y1 - 5)), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 0), 2)
    
    plt.figure(figsize=(9, 6))
    plt.imshow(img)
    plt.title(f"Sample: {img_path.name}")
    plt.axis('off')
    plt.tight_layout()
    plt.show()

sample_img = image_files[0]
sample_ann = LABELS_DIR / f"{sample_img.stem}.txt"
show_sample(sample_img, sample_ann)

## 4. Agricultural Image Preprocessing
Field images often have varying illumination and soil contrast. We test CLAHE and vegetation indices (Excess Green index, ExG = 2G - R - B) to inspect canopy contrast.

In [ ]:
def apply_clahe(img_rgb, clip_limit=2.0):
    lab = cv2.cvtColor(img_rgb, cv2.COLOR_RGB2LAB)
    l, a, b = cv2.split(lab)
    clahe = cv2.createCLAHE(clipLimit=clip_limit, tileGridSize=(8, 8))
    l_eq = clahe.apply(l)
    return cv2.cvtColor(cv2.merge((l_eq, a, b)), cv2.COLOR_LAB2RGB)

def compute_exg(img_rgb):
    f = img_rgb.astype(np.float32) / 255.0
    r, g, b = f[:, :, 0], f[:, :, 1], f[:, :, 2]
    return 2.0 * g - r - b

raw_img = cv2.cvtColor(cv2.imread(str(image_files[0])), cv2.COLOR_BGR2RGB)
clahe_img = apply_clahe(raw_img)
exg_map = compute_exg(raw_img)

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
axes[0].imshow(raw_img)
axes[0].set_title("Original RGB")
axes[0].axis('off')

axes[1].imshow(clahe_img)
axes[1].set_title("CLAHE Enhanced")
axes[1].axis('off')

im3 = axes[2].imshow(exg_map, cmap='YlGn')
axes[2].set_title("Excess Green Index (ExG)")
axes[2].axis('off')
plt.colorbar(im3, ax=axes[2], fraction=0.046, pad=0.04)

plt.tight_layout()
plt.show()

## 5. Model Training
Train YOLO11 Nano (`yolo11n.pt`) with standard agricultural augmentations (mosaic, rotation, flips).
Resumes automatically if a checkpoint exists in `runs/detect/mhweed16_yolo11n/weights/last.pt`.

In [ ]:
epochs = 50
batch = 8
imgsz = 640
run_name = "mhweed16_yolo11n"

run_dir = ROOT_DIR / "runs" / "detect" / run_name
last_weights = run_dir / "weights" / "last.pt"
best_weights = run_dir / "weights" / "best.pt"

if last_weights.exists():
    print(f"Resuming from checkpoint: {last_weights}")
    model = YOLO(str(last_weights))
    results = model.train(resume=True)
else:
    print(f"Starting fresh training for {epochs} epochs on {device}...")
    model = YOLO("yolo11n.pt")
    results = model.train(
        data="mhweed16.yaml",
        epochs=epochs,
        batch=batch,
        imgsz=imgsz,
        device=device,
        workers=2,
        name=run_name,
        exist_ok=True,
        amp=False,  # Set False for Intel Arc XPU stability
        patience=15,
        lr0=0.01,
        lrf=0.01,
        degrees=45.0,
        flipud=0.5,
        fliplr=0.5,
        mosaic=1.0
    )

## 6. Evaluation & Test Set Metrics
Evaluate the best checkpoint on the held-out test split (`test.txt`).

In [ ]:
eval_weights = best_weights if best_weights.exists() else pathlib.Path("yolo11n.pt")
eval_model = YOLO(str(eval_weights))

metrics = eval_model.val(
    data="mhweed16.yaml",
    split="test",
    device=device,
    batch=batch,
    imgsz=imgsz
)

print("\n--- Test Set Metrics ---")
print(f"mAP50:      {metrics.box.map50:.4f}")
print(f"mAP50-95:   {metrics.box.map:.4f}")
print(f"Precision:  {metrics.box.mp:.4f}")
print(f"Recall:     {metrics.box.mr:.4f}")

## 7. Inference Visualization
Compare ground truth bounding boxes side-by-side with YOLO11 model predictions on test images.

In [ ]:
with open("test.txt", 'r', encoding='utf-8') as f:
    test_image_paths = [pathlib.Path(line.strip()) for line in f if line.strip()]

selected_samples = random.sample(test_image_paths, min(3, len(test_image_paths)))

fig, axes = plt.subplots(len(selected_samples), 2, figsize=(14, 5 * len(selected_samples)))
if len(selected_samples) == 1:
    axes = np.array([axes])

for idx, test_p in enumerate(selected_samples):
    raw = cv2.imread(str(test_p))
    gt_img = cv2.cvtColor(raw.copy(), cv2.COLOR_BGR2RGB)
    h, w, _ = gt_img.shape
    
    ann_p = LABELS_DIR / f"{test_p.stem}.txt"
    if ann_p.exists():
        with open(ann_p, 'r', encoding='utf-8') as f:
            for line in f:
                parts = line.strip().split()
                if not parts:
                    continue
                cid = int(parts[0])
                xc, yc, bw, bh = map(float, parts[1:5])
                x1, y1 = int((xc - bw/2) * w), int((yc - bh/2) * h)
                x2, y2 = int((xc + bw/2) * w), int((yc + bh/2) * h)
                cv2.rectangle(gt_img, (x1, y1), (x2, y2), (0, 255, 0), 2)
                cv2.putText(gt_img, CLASS_NAMES.get(cid, str(cid)), (x1, max(20, y1-5)),
                            cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 0), 2)
    
    # Prediction
    pred_res = eval_model.predict(source=str(test_p), imgsz=imgsz, device=device, conf=0.25, verbose=False)[0]
    pred_img = cv2.cvtColor(pred_res.plot(), cv2.COLOR_BGR2RGB)
    
    axes[idx, 0].imshow(gt_img)
    axes[idx, 0].set_title(f"Ground Truth ({test_p.name})")
    axes[idx, 0].axis('off')
    
    axes[idx, 1].imshow(pred_img)
    axes[idx, 1].set_title(f"YOLO11 Prediction ({test_p.name})")
    axes[idx, 1].axis('off')

plt.tight_layout()
plt.show()

## 8. Export Model
Export trained weights to ONNX format for deployment.

In [ ]:
if best_weights.exists():
    try:
        onnx_path = eval_model.export(format="onnx", dynamic=True)
        print(f"Exported ONNX model to: {onnx_path}")
    except Exception as e:
        print(f"Export note: {e}")